In [0]:
from pyspark.sql.functions import *
import os
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from pyspark.sql.types import TimestampType, StructType, StructField, ArrayType, DoubleType, IntegerType
import pandas as pd
from pyarrow import *
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression  
from pyspark.ml.feature import *
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
import pyarrow.parquet as pq


In [0]:
lreg_df = spark.read.table("hive_metastore.default.lreg_df")

In [0]:
display(lreg_df.select("ID").distinct().count())

#### LREG

In [0]:
final_df = lreg_df.drop("AO","NOME","TIPOINST","TAGCOM","REDE","ID_prefix","ID_OBJECTO")

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.sql.functions import when, col

# AM/PM to 0/1
df = lreg_df.withColumn("AMPM_flag", when(col("AM_PM")=="PM", 1.0).otherwise(0.0))

# Index + OHE for ID & CONCELHO
id_indexer        = StringIndexer(inputCol="ID",        outputCol="ID_idx",        handleInvalid="keep")
id_ohe            = OneHotEncoder(inputCols=["ID_idx"], outputCols=["ID_ohe"],     dropLast=True)

concelho_indexer  = StringIndexer(inputCol="CONCELHO",  outputCol="CONCELHO_idx",  handleInvalid="keep")
concelho_ohe      = OneHotEncoder(inputCols=["CONCELHO_idx"], outputCols=["CONCELHO_ohe"], dropLast=True)

numeric_cols = [
    "INTENSITY","TENSION","H_LIM_I","H_LIM_T",
    "MAVERAGE_2H_I","MAVERAGE_2H_T","MAVERAGE_1D_I","MAVERAGE_1D_T",
    "EVENT_COUNT_I","EVENT_COUNT_T","TIME_OVER_LIMIT_I","TIME_OVER_LIMIT_T",
    "DAY_OF_WEEK","DAY_OF_MONTH","DAY_OF_YEAR","HOUR_OF_DAY",
    "temperature","humidity","wind_speed","precipitation",
    "AMPM_flag"
]

assembler_num  = VectorAssembler(inputCols=numeric_cols, outputCol="num_vec")
scaler_num     = StandardScaler(inputCol="num_vec", outputCol="num_scaled",
                                withMean=False, withStd=True)  # don't center sparse
assembler_all  = VectorAssembler(inputCols=["num_scaled"] + ["ID_ohe", "CONCELHO_ohe"], outputCol="features")



lr = LogisticRegression(featuresCol="features", labelCol="has_falha")  # tune as needed

pipeline = Pipeline(stages=[id_indexer, id_ohe, concelho_indexer, concelho_ohe, assembler_num, scaler_num, assembler_all, lr])

In [0]:
# final_df = final_df.withColumn(
#     "weight",
#     when(col("has_falha") == 1, 0.5).otherwise(0.05)  # tune this ratio
# )

In [0]:
# # choose hash dimensions (tune if needed)
# HASH_DIM_ID       = 2**15   # 32,768
# HASH_DIM_CONCELHO = 2**14   # 16,384

# id_hasher = FeatureHasher(
#     inputCols=[ID_COL], outputCol=ID_HASH_COL, numFeatures=HASH_DIM_ID
# )
# cc_hasher = FeatureHasher(
#     inputCols=[CC_COL], outputCol=CC_HASH_COL, numFeatures=HASH_DIM_CONCELHO
# )


In [0]:
# assembler = VectorAssembler(
#     inputCols = numeric_cols + [ID_HASH_COL, CC_HASH_COL, AMPM_OHE_COL],
#     outputCol = FEATURES_COL
# )


In [0]:
# scaler = StandardScaler(
#     inputCol=FEATURES_COL, outputCol=SCALED_COL, withStd=True, withMean=False
# )


In [0]:
cutoff_date = "2023-11-30"
train = df.filter(col("DATE") < cutoff_date)
test = df.filter(col("DATE") >= cutoff_date)

In [0]:
# lr = LogisticRegression(featuresCol=SCALED_COL, labelCol=LABEL_COL)

In [0]:
# stages = [am_pm_indexer, am_pm_ohe, id_hasher, cc_hasher, assembler, scaler, lr]
# pipeline = Pipeline(stages=stages)

In [0]:
model = pipeline.fit(train)

In [0]:
predictions = model.transform(test)

display(predictions.select("has_falha", "prediction", "probability"))

In [0]:
evaluator = BinaryClassificationEvaluator(labelCol="has_falha", rawPredictionCol="rawPrediction")
print("AUC:", evaluator.evaluate(predictions))


In [0]:
confusion_df = (
    predictions.groupBy("has_falha", "prediction")
    .count()
    .orderBy("has_falha", "prediction")
)
display(confusion_df)

Fiz regressao logistica e a primeira run foi muito ma, estou a ver alternativas para melhorar os dados. Adicionar weight ou tentar oversampling

#### Analysis

In [0]:
# === Uniform, DBFS-safe evaluator (adds filename tag like "_weather") ===
import os, json, time, math, datetime as dt
import numpy as np

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from pyspark.sql import functions as F, Window as W
from pyspark.sql.types import *
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

def _counts_to_metrics(tp, fp, tn, fn):
    tot  = tp + fp + tn + fn
    acc  = (tp + tn) / tot if tot else 0.0
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec  = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    bal  = 0.5 * (rec + spec)
    f1   = (2 * prec * rec) / (prec + rec) if (prec + rec) else 0.0
    den  = math.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
    mcc  = ((tp*tn - fp*fn) / den) if den else 0.0
    return acc, prec, rec, spec, bal, f1, mcc

def _plot_cm(cm, title, path):
    fig, ax = plt.subplots(figsize=(4,4), dpi=160)
    im = ax.imshow(cm, interpolation="nearest")
    ax.set_title(title); fig.colorbar(im)
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(["Pred 0","Pred 1"])
    ax.set_yticklabels(["True 0","True 1"])
    thr = cm.max()/2 if cm.size else 0
    for (i,j), v in np.ndenumerate(cm):
        ax.text(j, i, f"{int(v):,}", ha="center", va="center",
                color="white" if v > thr else "black")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    fig.tight_layout(); os.makedirs(os.path.dirname(path), exist_ok=True)
    fig.savefig(path, bbox_inches="tight"); plt.close(fig)

def evaluate_predictions_spark(predictions,
                               labelCol="has_falha",
                               probCol="probability",
                               rawCol="rawPrediction",
                               model_name="logreg_v1",
                               report_base="dbfs:/reports",
                               delta_log_path="dbfs:/reports/metrics_delta",
                               file_tag="_weather"):   # <<--- NEW
    # normalize tag ("" or startswith "_")
    file_tag = "" if not file_tag else (file_tag if file_tag.startswith("_") else f"_{file_tag}")

    dbutils.fs.mkdirs(report_base)
    dbutils.fs.mkdirs(delta_log_path)

    # 1) Extract p(class=1)
    df = (predictions
          .withColumn("_prob_arr", vector_to_array(F.col(probCol)))
          .withColumn("proba", F.col("_prob_arr")[1].cast(DoubleType()))
          .withColumn("label_int", F.col(labelCol).cast(IntegerType()))
          .select("label_int", "proba", rawCol))
    df = df.filter(F.col("proba").isNotNull() & F.col("label_int").isNotNull())

    # Totals
    P, N = df.select(
        F.sum(F.when(F.col("label_int")==1, 1).otherwise(0)),
        F.sum(F.when(F.col("label_int")==0, 1).otherwise(0))
    ).first()
    P, N = int(P), int(N)

    # 2) AUC / AUPRC on original predictions
    auc  = float(BinaryClassificationEvaluator(labelCol=labelCol, rawPredictionCol=rawCol, metricName="areaUnderROC").evaluate(predictions))
    aupr = float(BinaryClassificationEvaluator(labelCol=labelCol, rawPredictionCol=rawCol, metricName="areaUnderPR").evaluate(predictions))

    # 3) Confusion @ 0.5
    df05 = df.withColumn("pred05", (F.col("proba") >= 0.5).cast("int"))
    tp05, fp05, fn05, tn05 = df05.select(
        F.sum(F.when((F.col("pred05")==1) & (F.col("label_int")==1), 1).otherwise(0)),
        F.sum(F.when((F.col("pred05")==1) & (F.col("label_int")==0), 1).otherwise(0)),
        F.sum(F.when((F.col("pred05")==0) & (F.col("label_int")==1), 1).otherwise(0)),
        F.sum(F.when((F.col("pred05")==0) & (F.col("label_int")==0), 1).otherwise(0))
    ).first()
    tp05, fp05, fn05, tn05 = map(int, (tp05, fp05, fn05, tn05))
    acc05, prec05, rec05, spec05, bal05, f105, mcc05 = _counts_to_metrics(tp05, fp05, tn05, fn05)

    # 4) Best-F1 threshold via bucketing
    dfb = df.withColumn("bucket", (F.floor(F.col("proba") * 1000) / 1000.0).cast(DoubleType()))
    byb = (dfb.groupBy("bucket")
              .agg(F.sum(F.when(F.col("label_int")==1, 1).otherwise(0)).alias("pos"),
                   F.sum(F.when(F.col("label_int")==0, 1).otherwise(0)).alias("neg")))
    w = W.orderBy(F.col("bucket").desc())
    byb = (byb
           .withColumn("cum_pos", F.sum("pos").over(w))
           .withColumn("cum_neg", F.sum("neg").over(w))
           .withColumn("tp", F.col("cum_pos"))
           .withColumn("fp", F.col("cum_neg"))
           .withColumn("fn", F.lit(P) - F.col("cum_pos"))
           .withColumn("tn", F.lit(N) - F.col("cum_neg"))
           .withColumn("prec", F.when(F.col("tp")+F.col("fp")>0, F.col("tp")/(F.col("tp")+F.col("fp"))).otherwise(F.lit(0.0)))
           .withColumn("rec",  F.when(F.col("tp")+F.col("fn")>0, F.col("tp")/(F.col("tp")+F.col("fn"))).otherwise(F.lit(0.0)))
           .withColumn("f1",   F.when(F.col("prec")+F.col("rec")>0, 2*F.col("prec")*F.col("rec")/(F.col("prec")+F.col("rec"))).otherwise(F.lit(0.0))))
    best = byb.orderBy(F.col("f1").desc(), F.col("bucket").desc()).limit(1).first()
    best_thr = float(best["bucket"])
    tpB, fpB, fnB, tnB = map(int, (best["tp"], best["fp"], best["fn"], best["tn"]))
    accB, precB, recB, specB, balB, f1B, mccB = _counts_to_metrics(tpB, fpB, tnB, fnB)

    # 5) Save plots/JSON to /tmp then copy to DBFS (filenames tagged)
    ts = time.strftime("%Y%m%d_%H%M%S")
    local_dir = f"/tmp/{model_name}_{ts}"
    os.makedirs(local_dir, exist_ok=True)

    cm05   = np.array([[tn05, fp05],[fn05, tp05]])
    cmbest = np.array([[tnB,  fpB ],[fnB,  tpB ]])

    cm05_name   = f"cm_thr_0.50{file_tag}.png"
    cmbest_name = f"cm_thr_{best_thr:.3f}{file_tag}.png"
    report_name = f"report{file_tag}.json"

    cm05_local   = os.path.join(local_dir, cm05_name)
    cmbest_local = os.path.join(local_dir, cmbest_name)
    _plot_cm(cm05,   f"{model_name} | thr=0.50", cm05_local)
    _plot_cm(cmbest, f"{model_name} | thr={best_thr:.3f}", cmbest_local)

    db_dir = f"{report_base}/{model_name}/{ts}"
    dbutils.fs.mkdirs(db_dir)
    dbutils.fs.cp(f"file:{cm05_local}",   f"{db_dir}/{cm05_name}",   True)
    dbutils.fs.cp(f"file:{cmbest_local}", f"{db_dir}/{cmbest_name}", True)

    summary = {
        "model_name": model_name,
        "timestamp": ts,
        "n_samples": int(P+N),
        "pos_rate": float(P/(P+N)) if (P+N) else 0.0,
        "auc": auc, "ap": aupr,
        "thr_fixed": 0.5,
        "metrics_fixed": {"tp":tp05,"fp":fp05,"tn":tn05,"fn":fn05,
                          "acc":acc05,"prec":prec05,"rec":rec05,"f1":f105,"spec":spec05,"bal_acc":bal05,"mcc":mcc05},
        "thr_bestF1": best_thr,
        "metrics_bestF1": {"tp":tpB,"fp":fpB,"tn":tnB,"fn":fnB,
                           "acc":accB,"prec":precB,"rec":recB,"f1":f1B,"spec":specB,"bal_acc":balB,"mcc":mccB},
        "confusion_fixed_path": f"{db_dir}/{cm05_name}",
        "confusion_best_path":  f"{db_dir}/{cmbest_name}"
    }
    json_local = os.path.join(local_dir, report_name)
    with open(json_local, "w") as f:
        json.dump(summary, f, indent=2)
    dbutils.fs.cp(f"file:{json_local}", f"{db_dir}/{report_name}", True)

    # 6) Append metrics to a Delta table (unchanged)
    schema = StructType([
        StructField("model", StringType()), StructField("ts_utc", StringType()),
        StructField("threshold", DoubleType()),
        StructField("auc", DoubleType()), StructField("auprc", DoubleType()),
        StructField("accuracy", DoubleType()), StructField("f1", DoubleType()),
        StructField("precision", DoubleType()), StructField("recall", DoubleType()),
        StructField("specificity", DoubleType()), StructField("balanced_accuracy", DoubleType()),
        StructField("mcc", DoubleType()),
        StructField("tp", LongType()), StructField("fp", LongType()),
        StructField("tn", LongType()), StructField("fn", LongType()),
        StructField("n_samples", LongType()),
    ])
    now = dt.datetime.utcnow().isoformat()
    rows = [
        (model_name, now, 0.5,      auc, aupr, acc05, f105, prec05, rec05, spec05, bal05, mcc05, tp05, fp05, tn05, fn05, int(P+N)),
        (model_name, now, best_thr, auc, aupr, accB, f1B,  precB,  recB,  specB,  balB,  mccB,  tpB,  fpB,  tnB,  fnB,  int(P+N)),
    ]
    spark.createDataFrame(rows, schema)\
         .write.format("delta").mode("append").save(delta_log_path)

    print(f"=== {model_name} | {ts} ===")
    print(f"N={int(P+N)}, Pos rate={P/(P+N):.4f}")
    print(f"AUC={auc:.4f}, AUPRC={aupr:.4f}")
    print(f"[thr=0.50]   ACC={acc05:.4f}  PREC={prec05:.4f}  REC={rec05:.4f}  F1={f105:.4f}")
    print(f"[thr={best_thr:.3f}] ACC={accB:.4f}  PREC={precB:.4f}  REC={recB:.4f}  F1={f1B:.4f}")
    print("Saved:", f"{db_dir}/{cm05_name}")
    print("Saved:", f"{db_dir}/{cmbest_name}")
    print("Report:", f"{db_dir}/{report_name}")
    print("Logged metrics to Delta:", delta_log_path)

    return {
        "report_dir": db_dir,
        "best_thr": best_thr,
        "auc": auc, "auprc": aupr,
        "fixed": {"acc":acc05,"prec":prec05,"rec":rec05,"f1":f105},
        "bestF1":{"acc":accB,"prec":precB,"rec":recB,"f1":f1B},
        "cm_fixed_path": f"{db_dir}/{cm05_name}",
        "cm_best_path":  f"{db_dir}/{cmbest_name}",
        "report_path":   f"{db_dir}/{report_name}"
    }


In [0]:
# predictions = lr_model.transform(test_df)
res = evaluate_predictions_spark(
    predictions,
    labelCol="has_falha",
    probCol="probability",
    rawCol="rawPrediction",
    model_name="logreg_v1",
    file_tag="_weather"        # <<— adds suffix to PNGs + JSON
)
from PIL import Image
from IPython.display import display

display(Image.open(res["cm_fixed_path"].replace("dbfs:/", "/dbfs/")))
display(Image.open(res["cm_best_path"].replace("dbfs:/", "/dbfs/")))


In [0]:
import builtins
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyspark.ml import PipelineModel
from pyspark.ml.classification import LogisticRegressionModel, OneVsRestModel
from pyspark.ml.feature import VectorAssembler

def spark_logreg_feature_importance(model_or_pipeline,
                                    df_with_features,
                                    featuresCol="features",
                                    model_name="logreg",
                                    top_k=30,
                                    out_base_dbfs="dbfs:/tmp/feature_importance",
                                    file_tag="_weather"):   # <— NEW
    """
    Extract LR coefficients as feature importances.
    Works with a fitted LogisticRegressionModel or a PipelineModel that contains it.
    Files are saved with a suffix (e.g., '_weather'): importance_weather.csv, importance_top_weather.png
    """

    # normalize tag
    file_tag = "" if not file_tag else (file_tag if file_tag.startswith("_") else f"_{file_tag}")

    # ---- 1) Find the fitted LR model and feature names
    lr_model = None
    feat_names = None

    if isinstance(model_or_pipeline, LogisticRegressionModel):
        lr_model = model_or_pipeline
    elif isinstance(model_or_pipeline, PipelineModel):
        # find LR model among stages (search from end)
        for st in reversed(model_or_pipeline.stages):
            if isinstance(st, LogisticRegressionModel):
                lr_model = st
                break
            if isinstance(st, OneVsRestModel):
                for m in st.models:
                    if isinstance(m, LogisticRegressionModel):
                        lr_model = m
                        break
        # recover feature names from VectorAssembler that outputs featuresCol
        for st in model_or_pipeline.stages:
            if isinstance(st, VectorAssembler) and st.getOutputCol() == featuresCol:
                feat_names = list(st.getInputCols())
                break
    else:
        raise TypeError("Pass a fitted LogisticRegressionModel or a PipelineModel.")

    if lr_model is None:
        raise ValueError("Could not locate a fitted LogisticRegressionModel in the object provided.")

    # If we still don't have names, fallback to generic f0..fN-1 using the vector size
    vec_size = int(lr_model.coefficients.size)
    if not feat_names or len(feat_names) != vec_size:
        feat_names = [f"f{i}" for i in range(vec_size)]

    # ---- 2) Coefficients → importance table
    coefs = np.array(lr_model.coefficients.toArray())
    df = pd.DataFrame({
        "feature": feat_names,
        "coef": coefs,
        "abs_coef": np.abs(coefs),
        "direction": np.where(coefs >= 0, "↑ risk", "↓ risk")
    }).sort_values("abs_coef", ascending=False).reset_index(drop=True)

    # ---- 3) Save CSV + quick bar plot (tagged filenames)
    ts = time.strftime("%Y%m%d_%H%M%S")
    out_dir_dbfs = f"{out_base_dbfs}/{model_name}/{ts}"
    out_dir_driver = "/dbfs" + out_dir_dbfs[len("dbfs:"):]   # convert to driver FS
    os.makedirs(out_dir_driver, exist_ok=True)

    csv_name = f"importance{file_tag}.csv"
    png_name = f"importance_top{file_tag}.png"

    csv_path = os.path.join(out_dir_driver, csv_name)
    df.to_csv(csv_path, index=False)

    top = df.head(top_k).iloc[::-1]  # reverse for horizontal plot
    height = builtins.max(4.0, 0.25*len(top))
    plt.figure(figsize=(8, height))
    plt.barh(top["feature"], top["abs_coef"])
    plt.title(f"{model_name} — top {len(top)} | |coef|")
    plt.xlabel("|coef|")
    plt.tight_layout()
    png_path = os.path.join(out_dir_driver, png_name)
    plt.savefig(png_path, dpi=160, bbox_inches="tight"); plt.close()

    return {
        "df": df,
        "csv_path": csv_path,         # driver path (usable with pandas)
        "png_path": png_path,         # driver path (usable with PIL)
        "out_dir_dbfs": out_dir_dbfs  # DBFS directory
    }


In [0]:
res_imp = spark_logreg_feature_importance(
    model_or_pipeline=model,
    df_with_features=predictions,      # not used, kept for interface symmetry
    featuresCol="features",
    model_name="logreg_v1",
    file_tag="_weather"                # <— adds suffix
)

# Preview PNG in notebook
from PIL import Image
from IPython.display import display
display(Image.open(res_imp["png_path"]))


In [0]:
# Top coefficients table
res_imp["df"].head(20)

# Bar chart of top |coef|
from PIL import Image
from IPython.display import display
display(Image.open(res_imp["png_path"]))


In [0]:
import os, time, builtins
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from pyspark.ml import PipelineModel
from pyspark.ml.classification import LogisticRegressionModel
from pyspark.ml.feature import StringIndexerModel, OneHotEncoderModel, VectorAssembler


In [0]:
def _find_stage(model, typ):
    if isinstance(model, PipelineModel):
        for st in model.stages:
            if isinstance(st, typ):
                return st
    return model if isinstance(model, typ) else None

def _find_all(model, typ):
    if not isinstance(model, PipelineModel): return []
    return [st for st in model.stages if isinstance(st, typ)]

def _labels_for_indexer(model, output_col):
    for st in _find_all(model, StringIndexerModel):
        if st.getOutputCol() == output_col:
            return list(st.labels)
    return None

def _drop_last_for_ohe(model, output_col):
    # works for OneHotEncoderModel (single or multi); default True if not found
    for st in _find_all(model, OneHotEncoderModel):
        outs = st.getOutputCols() if hasattr(st, "getOutputCols") else [st.getOutputCol()]
        if output_col in outs:
            # Spark keeps one dropLast for all inputs; default True if missing
            try:
                return bool(st.getDropLast())
            except:
                return True
    return True


In [0]:
# --- Helpers to build names correctly for OHE + numeric pipelines ---

import builtins
import numpy as np
import pandas as pd

from pyspark.ml import PipelineModel
from pyspark.ml.classification import LogisticRegressionModel
from pyspark.ml.feature import (
    VectorAssembler, StandardScalerModel, OneHotEncoderModel, StringIndexerModel
)

def _find_lr_and_feature_col(pipeline_model: PipelineModel):
    lr = None
    for st in reversed(pipeline_model.stages):
        if isinstance(st, LogisticRegressionModel):
            lr = st
            break
    if lr is None:
        raise ValueError("No LogisticRegressionModel found in PipelineModel.")
    return lr, lr.getFeaturesCol()

def _find_source_vector_inputs(pipeline_model: PipelineModel, featuresCol: str):
    """
    Returns (inputs_list, source_stage) for the vector that finally feeds LR.
    Handles an optional StandardScaler in front of LR.
    """
    # find stage that outputs featuresCol
    src = None
    for st in reversed(pipeline_model.stages):
        if hasattr(st, "getOutputCol") and st.getOutputCol() == featuresCol:
            src = st
            break
    if isinstance(src, StandardScalerModel):
        vec_col = src.getInputCol()
        # find assembler that outputs vec_col
        asm = None
        for st in pipeline_model.stages:
            if isinstance(st, VectorAssembler) and st.getOutputCol() == vec_col:
                asm = st
                break
        if asm is None:
            raise ValueError(f"Could not find VectorAssembler that outputs '{vec_col}'.")
        return list(asm.getInputCols()), asm
    elif isinstance(src, VectorAssembler):
        return list(src.getInputCols()), src
    else:
        # fallback: look for any assembler feeding into featuresCol indirectly
        for st in pipeline_model.stages:
            if isinstance(st, VectorAssembler) and st.getOutputCol() == featuresCol:
                return list(st.getInputCols()), st
        raise ValueError(f"Could not resolve inputs for featuresCol='{featuresCol}'.")

def _find_stage_by_output(pipeline_model: PipelineModel, out_col: str, cls):
    for st in pipeline_model.stages:
        if isinstance(st, cls):
            # OneHotEncoderModel can be multi-column; check both APIs
            if hasattr(st, "getOutputCols") and out_col in (st.getOutputCols() or []):
                return st
            if hasattr(st, "getOutputCol") and st.getOutputCol() == out_col:
                return st
    return None

def _find_indexer_model(pipeline_model: PipelineModel, out_idx_col: str):
    # StringIndexerModel outputs the indexed col
    for st in pipeline_model.stages:
        if isinstance(st, StringIndexerModel) and st.getOutputCol() == out_idx_col:
            return st
    return None

def _expand_ohe_names(prefix: str, ohe_model: OneHotEncoderModel, ohe_out_col: str,
                      indexer_model: StringIndexerModel):
    """
    Build readable names for a single OHE output column using the fitted models.
    """
    out_cols = ohe_model.getOutputCols() or [ohe_model.getOutputCol()]
    if ohe_out_col not in out_cols:
        raise ValueError(f"OHE model does not output '{ohe_out_col}'.")
    j = out_cols.index(ohe_out_col)
    cat_size = int(ohe_model.categorySizes[j])  # total categories seen (incl. 'unseen' if indexing used keep)
    drop_last = bool(ohe_model.getDropLast())

    labels = list(indexer_model.labels)  # known categories, no 'unseen' here
    # account for handleInvalid="keep" in indexer → an extra unseen bucket at index len(labels)
    handle_keep = (indexer_model.getHandleInvalid() or "").lower() == "keep"
    total_cats = len(labels) + (1 if handle_keep else 0)

    # The encoder’s categorySizes is the source of truth:
    # number of OHE dims produced = cat_size - (1 if drop_last else 0)
    ohe_dims = cat_size - (1 if drop_last else 0)

    names = []
    # produce human labels for as many as we can:
    # if drop_last=True, highest index is dropped → we include indices [0 .. cat_size-2]
    max_onehot_index = cat_size - (1 if drop_last else 0) - 1

    # Known labels first
    take_known = builtins.min(len(labels), max_onehot_index + 1)
    names.extend([f"{prefix}=={lab}" for lab in labels[:take_known]])

    # If there is an 'unseen' bucket AND it isn't the dropped one, add a name for it
    if handle_keep and (len(labels) <= max_onehot_index):
        names.append(f"{prefix}==__unseen__")

    # pad remaining (rare) with generic names to match ohe_dims
    while len(names) < ohe_dims:
        names.append(f"{prefix}::extra_{len(names)}")

    return names, [prefix] * ohe_dims

def build_feature_names_from_pipeline(pipeline_model: PipelineModel,
                                      featuresCol: str = None,
                                      id_ohe_col: str = "ID_ohe",
                                      concelho_ohe_col: str = "CONCELHO_ohe"):
    lr, lr_features = _find_lr_and_feature_col(pipeline_model)
    if featuresCol is None:
        featuresCol = lr_features

    inputs, src_stage = _find_source_vector_inputs(pipeline_model, featuresCol)

    # locate OHE models and their paired indexers for ID / CONCELHO
    id_ohe  = _find_stage_by_output(pipeline_model, id_ohe_col, OneHotEncoderModel)
    cc_ohe  = _find_stage_by_output(pipeline_model, concelho_ohe_col, OneHotEncoderModel)

    # If your pipeline used StringIndexer -> *_idx -> OHE, the indexer outputs *_idx
    id_indexer = _find_indexer_model(pipeline_model, out_idx_col="ID_idx")
    cc_indexer = _find_indexer_model(pipeline_model, out_idx_col="CONCELHO_idx")

    names, groups = [], []
    for c in inputs:
        if c == id_ohe_col and id_ohe and id_indexer:
            nms, grp = _expand_ohe_names("ID", id_ohe, id_ohe_col, id_indexer)
            names += nms; groups += grp
        elif c == concelho_ohe_col and cc_ohe and cc_indexer:
            nms, grp = _expand_ohe_names("CONCELHO", cc_ohe, concelho_ohe_col, cc_indexer)
            names += nms; groups += grp
        else:
            # numeric passthrough (or already prepared single feature)
            names.append(c); groups.append(c)

    # Final sanity guard: align to coefficient length
    coef_len = int(lr.coefficients.size)
    if len(names) < coef_len:
        # pad with placeholders (should not happen if pipeline discovered correctly)
        pad_n = coef_len - len(names)
        names += [f"__extra_{i}" for i in range(pad_n)]
        groups += [ "__extra" ] * pad_n
    elif len(names) > coef_len:
        names = names[:coef_len]
        groups = groups[:coef_len]

    return names, groups, lr


In [0]:
import os, time, matplotlib.pyplot as plt
import builtins


def logreg_ohe_feature_importance(pipeline_model,
                                  featuresCol: str = None,
                                  id_ohe_col="ID_ohe",
                                  concelho_ohe_col="CONCELHO_ohe",
                                  model_name="logreg_ohe_v1",
                                  out_base_dbfs="dbfs:/reports_ohe",
                                  top_k=40):
    feature_names, groups, lr = build_feature_names_from_pipeline(
        pipeline_model, featuresCol=featuresCol,
        id_ohe_col=id_ohe_col, concelho_ohe_col=concelho_ohe_col
    )

    coefs = np.array(lr.coefficients.toArray())
    if len(feature_names) != len(coefs):
        raise ValueError(f"Still mismatched: {len(feature_names)} vs {len(coefs)}")

    df = pd.DataFrame({
        "feature": feature_names,
        "group": groups,
        "coef": coefs,
        "abs_coef": np.abs(coefs)
    }).sort_values("abs_coef", ascending=False).reset_index(drop=True)

    # grouped views
    grouped = (df.groupby("group")
                 .agg(n_dims=("coef","size"),
                      l2=("coef", lambda x: float(np.sqrt(np.sum(x**2)))),
                      l1=("abs_coef", "sum"))
                 .reset_index())
    grouped["rms_coef"] = grouped["l2"] / grouped["n_dims"].clip(lower=1)
    grouped["mean_abs"] = grouped["l1"] / grouped["n_dims"].clip(lower=1)

    # save to DBFS (via local-copy)
    ts = time.strftime("%Y%m%d_%H%M%S")
    out_dir_dbfs   = f"{out_base_dbfs}/{model_name}/{ts}"
    out_dir_driver = "/dbfs" + out_dir_dbfs[len("dbfs:"):]
    os.makedirs(out_dir_driver, exist_ok=True)

    all_csv   = os.path.join(out_dir_driver, "importance_all.csv")
    group_csv = os.path.join(out_dir_driver, "importance_grouped.csv")
    df.to_csv(all_csv, index=False)
    grouped.to_csv(group_csv, index=False)

    # plot top_k
    top = df.head(top_k).iloc[::-1]
    plt.figure(figsize=(10, builtins.max(4.0, 0.30*len(top))))
    plt.barh(top["feature"], top["abs_coef"])
    plt.title(f"{model_name} — top {len(top)} | |coef|")
    plt.xlabel("|coef|"); plt.tight_layout()
    top_png = os.path.join(out_dir_driver, "importance_top.png")
    plt.savefig(top_png, dpi=160, bbox_inches="tight"); plt.close()

    print("Saved:", all_csv.replace("/dbfs","dbfs:"))
    print("Saved:", group_csv.replace("/dbfs","dbfs:"))
    print("Saved:", top_png.replace("/dbfs","dbfs:"))

    return {"df": df, "grouped": grouped,
            "csv_path": all_csv, "grouped_csv_path": group_csv, "png_path": top_png,
            "out_dir_dbfs": out_dir_dbfs}


In [0]:
imp = logreg_ohe_feature_importance(
    model,                       # your fitted PipelineModel
    featuresCol=None,            # or "scaled_features" if LR uses that
    id_ohe_col="ID_ohe",
    concelho_ohe_col="CONCELHO_ohe",
    model_name="logreg_ohe_v1",
    out_base_dbfs="dbfs:/reports_ohe",
    top_k=40          
)


In [0]:
import os


run_dbfs = "dbfs:/reports_ohe/logreg_ohe_v1/20250916_180207"

# Convert DBFS URI -> driver path
run_dir = "/dbfs" + run_dbfs[len("dbfs:"):]
run_dir


In [0]:
import os, builtins
import pandas as pd
from IPython.display import display

# <-- make sure run_dir is correct, e.g. run_dir = "/dbfs/reports_ohe/logreg_ohe_v1/20250916_180207"
print("Grouped columns:", pd.read_csv(os.path.join(run_dir, "importance_grouped.csv")).columns.tolist())


In [0]:
import pandas as pd

all_df     = pd.read_csv(os.path.join(run_dir, "importance_all.csv"))
grouped_df = pd.read_csv(os.path.join(run_dir, "importance_grouped.csv"))

display(grouped_df.sort_values("l2", ascending=False))


In [0]:
import numpy as np, pandas as pd
g = grouped_df.copy()
g["rms_coef"]   = g["l2"] / np.sqrt(g["n_dims"])
g["mean_abs"]   = g["l1"] / g["n_dims"]
g.sort_values("rms_coef", ascending=False).head(24)


In [0]:
from PIL import Image
from IPython.display import display

png_path = os.path.join(run_dir, "importance_top.png")
display(Image.open(png_path))


In [0]:
import builtins
import matplotlib.pyplot as plt

g = grouped_df.sort_values("l2", ascending=False).head(20).iloc[::-1]
plt.figure(figsize=(10, builtins.max(4, 0.35*len(g))))
plt.barh(g["group"], g["l2"])
plt.title("Grouped importance (L2 of coefficients)")
plt.xlabel("L2 norm")
plt.tight_layout()
plt.show()


In [0]:
# from pyspark.ml.linalg import VectorUDT

# def _extract_attr_names_from_metadata(field):
#     """
#     Try to get per-dimension names from field.metadata['ml_attr'] if present.
#     Returns a list of names or None.
#     """
#     md = dict(field.metadata) if hasattr(field, "metadata") else {}
#     ml_attr = md.get("ml_attr") or md.get("ml_attr.", None)  # be tolerant
#     if not isinstance(ml_attr, dict):
#         return None

#     attrs = ml_attr.get("attrs")
#     if not isinstance(attrs, dict):
#         return None

#     names = []
#     # attributes can be grouped under 'binary', 'nominal', 'numeric'
#     for group in ("binary", "nominal", "numeric"):
#         arr = attrs.get(group)
#         if isinstance(arr, list):
#             for item in arr:
#                 nm = item.get("name")
#                 # fallback if missing
#                 names.append(nm if nm is not None else f"attr_{len(names)}")
#     return names or None

# def _vector_size_from_row(df, col):
#     """Safely get vector size by reading a single row."""
#     v = df.select(col).limit(1).collect()[0][0]
#     return int(v.size) if v is not None else 0

# def build_feature_names_no_attribute(assembler, df_with_inputs, hash_dims=None):
#     """
#     Expand VectorAssembler inputCols into per-dimension names WITHOUT using pyspark.ml.attribute.
#     - numeric col -> ["col"]
#     - OHE/vector with metadata -> use metadata names when available
#     - hashed vector -> use provided hash_dims[col] to name as "col[i]"
#     """
#     if hash_dims is None:
#         hash_dims = {}

#     names = []
#     for col in assembler.getInputCols():
#         field = df_with_inputs.schema[col]
#         if isinstance(field.dataType, VectorUDT):
#             # 1) try metadata names (works for OneHotEncoder in many runtimes)
#             sub = _extract_attr_names_from_metadata(field)

#             # 2) if column is hashed, force names by declared dimension
#             if sub is None and col in hash_dims:
#                 size = int(hash_dims[col])
#                 sub = [f"{col}[{i}]" for i in range(size)]

#             # 3) last resort: get size from a tiny row and synth names
#             if sub is None:
#                 size = _vector_size_from_row(df_with_inputs, col)
#                 sub = [f"{col}[{i}]" for i in range(size)]
#         else:
#             sub = [col]

#         names.extend(sub)

#     return names


In [0]:
import re, builtins, numpy as np, pandas as pd
import matplotlib.pyplot as plt

def plot_grouped_importance(imp_df,
                            prefixes=None,
                            use="L2",           # "L2" or "L1"
                            normalize=False,    # if True, divide by sqrt(n_dims)
                            top_k=None,         # e.g., 30
                            title="Grouped feature importance",
                            out_png=None):
    """
    Group hashed buckets by original field (e.g., all 'ID_hv[xxxx]' -> 'ID (hashed)'),
    keep other (non-hashed) features as-is, and plot a single bar per group/feature.

    use="L2": sqrt(sum(coef^2)) per group   (good when groups have multiple dims)
    use="L1": sum(|coef|) per group
    normalize: divide score by sqrt(n_dims) to prevent large groups from dominating
    """

    if prefixes is None:
        prefixes = {"ID_hv[": "ID (hashed)", "CONCELHO_hv[": "CONCELHO (hashed)"}

    def group_label(name: str) -> str:
        for pfx, lbl in prefixes.items():
            if name.startswith(pfx):
                return lbl
        return name  # non-hashed features stay as their own “group”

    df = imp_df.copy()
    df["group"] = df["feature"].map(group_label)
    df["coef2"] = df["coef"] ** 2

    agg = (df.groupby("group", as_index=False)
             .agg(n_dims=("feature", "count"),
                  L1=("abs_coef", "sum"),
                  L2=("coef2", "sum")))
    agg["L2"] = np.sqrt(agg["L2"])

    score_col = "L2" if use.upper() == "L2" else "L1"
    agg["score"] = agg[score_col]
    if normalize:
        agg["score"] = agg["score"] / np.sqrt(agg["n_dims"].clip(lower=1))

    # Sort & keep top_k if requested
    agg = agg.sort_values("score", ascending=False)
    if top_k:
        agg = agg.head(top_k)

    # Plot
    agg_plot = agg.iloc[::-1]  # small→large for nice horizontal bars
    plt.figure(figsize=(10, builtins.max(4.0, 0.35 * len(agg_plot))))
    plt.barh(agg_plot["group"], agg_plot["score"])
    plt.title(title + (f"  ({use}{' / √dims' if normalize else ''})"))
    plt.xlabel("importance")
    plt.tight_layout()
    if out_png:
        plt.savefig(out_png, dpi=160, bbox_inches="tight")
        print("Saved:", out_png)
    plt.show()

    # Return the grouped table too
    cols = ["group", "n_dims", score_col, "score"]
    return agg[cols].reset_index(drop=True)


In [0]:
# Cell 1 — load importance_all.csv (or use in-memory)
import os, builtins
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    # from the previous importance run
    df_all = imp["df"].copy()
    out_dir_dbfs = imp["out_dir_dbfs"]
except Exception:
    # fallback: point this to the run you printed earlier
    out_dir_dbfs = "dbfs:/reports_ohe/logreg_ohe_v1/20250916_180207"  # <-- change me
    run_dir_driver = "/dbfs" + out_dir_dbfs[len("dbfs:"):]
    df_all = pd.read_csv(os.path.join(run_dir_driver, "importance_all.csv"))


In [0]:
# Cell 1 — load importance_all.csv (or use in-memory)
import os, builtins
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    # from the previous importance run
    df_all = imp["df"].copy()
    out_dir_dbfs = imp["out_dir_dbfs"]
except Exception:
    # fallback: point this to the run you printed earlier
    out_dir_dbfs = "dbfs:/reports_ohe/logreg_ohe_v1/20250916_180207"  # <-- change me
    run_dir_driver = "/dbfs" + out_dir_dbfs[len("dbfs:"):]
    df_all = pd.read_csv(os.path.join(run_dir_driver, "importance_all.csv"))


In [0]:
# Cell 3 — save CSV + barh plots to same run dir
run_dir_driver = "/dbfs" + out_dir_dbfs[len("dbfs:"):]
os.makedirs(run_dir_driver, exist_ok=True)

id_csv = os.path.join(run_dir_driver, "ID_top20.csv")
con_csv = os.path.join(run_dir_driver, "CONCELHO_top20.csv")
id_rows[["entity","coef","abs_coef"]].head(20).to_csv(id_csv, index=False)
con_rows[["entity","coef","abs_coef"]].head(20).to_csv(con_csv, index=False)

def barh(df, title, path, k=20):
    top = df.head(k).iloc[::-1]
    plt.figure(figsize=(10, builtins.max(4.0, 0.35*len(top))))
    plt.barh(top["entity"], top["abs_coef"])
    # annotate with signed coef
    for i, (v, s) in enumerate(zip(top["abs_coef"], top["coef"])):
        plt.text(v, i, f"  {'+' if s>=0 else '-'}{abs(s):.3f}", va='center')
    plt.title(title)
    plt.xlabel("|coef|")
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.close()

barh(id_rows, "LogReg (OHE) — top IDs by |coef|",
     os.path.join(run_dir_driver, "ID_top20.png"))
barh(con_rows, "LogReg (OHE) — top CONCELHOs by |coef|",
     os.path.join(run_dir_driver, "CONCELHO_top20.png"))

print("Saved:", id_csv.replace("/dbfs","dbfs:"))
print("Saved:", con_csv.replace("/dbfs","dbfs:"))
print("Saved:", os.path.join(out_dir_dbfs, "ID_top20.png"))
print("Saved:", os.path.join(out_dir_dbfs, "CONCELHO_top20.png"))
